In [1]:
!pip install transformers datasets torch pandas accelerate scikit-learn

In [31]:
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline

# 1. CSV Load Karein
df = pd.read_csv("patient_conversations.csv")

# 2. Label Encoding (0: High, 1: Low, 2: Medium)
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["urgency"])

# 3. Context Combine Karein (Symptom + Message)
df["text"] = "Symptom: " + df["symptom"].fillna("") + " | Message: " + df["patient_message"].fillna("")

# 4. Dataset & Tokenizer Setup
dataset = Dataset.from_pandas(df[["text", "label"]])
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# 5. Training Settings (10 Epochs for better learning)
training_args = TrainingArguments(
    output_dir="./my_custom_urgency_model",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    logging_steps=5,
    save_strategy="epoch"
)

# 6. Trainer Setup & Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Training shuru ho rahi hai...")
trainer.train()

# 7. Save Model
trainer.save_model("./my_custom_urgency_model")
tokenizer.save_pretrained("./my_custom_urgency_model")
print("Model fine-tune aur save ho gaya hai!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training shuru ho rahi hai...


Step,Training Loss
5,1.118394
10,1.083510
15,1.056163
20,1.037327
25,0.950623
30,0.878863
35,0.825404
40,1.072541
45,0.833630
50,0.826797


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model fine-tune aur save ho gaya hai!


In [37]:
classifier = pipeline("text-classification", model="./my_custom_urgency_model")

test_msg = input("Please elaborate your current condition: ")
result = classifier(test_msg)

# Correct Mapping (0: High, 1: Low, 2: Medium)
labels_map = {0: "High", 1: "Low", 2: "Medium"}
predicted_class = int(result[0]['label'].split('_')[-1])

print(f"\nPredicted Urgency Level : {labels_map[predicted_class]}")
print(f"Confidence Score        : {result[0]['score'] * 100:.1f}%")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Please elaborate your current condition: light itching on my forearm after applying a new moisturizer.

Predicted Urgency Level : Low
Confidence Score        : 43.1%
